# Deconvolution

### Imports


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import s3fs
import zarr

from deep_neurographs import img_utils
from scipy.ndimage import center_of_mass
from skimage import restoration
from tifffile import imwrite
from time import time

%matplotlib inline

IMG_PREFIX = {
    "653159": "exaSPIM_653159_2024-02-05_15-38-38_training-data",
    "657676": "exaSPIM_657676_2023-10-27_12-31-24_training-data",
    "667996": "exaSPIM_667996_2024-03-05_17-26-26_training-data",
    "671477": "exaSPIM_671477_2024-01-08_17-03-04_training-data",
    "674185": "exaSPIM_674185_2023-10-02_14-06-36_training-data",
    "667996": "exaSPIM_667996_2024-03-05_17-26-26_training-data",
    "675375": "exaSPIM_675375_2024-02-27_18-50-24_training-data",
    "676009": "exaSPIM_676009_2024-01-23_16-36-13_training-data",
    "706301": "exaSPIM_706301_2024-04-23_11-24-24_fusion_2024-05-21_00-00-03",
    "708369": "exaSPIM_708369_2024-04-08_15-20-36_fusion_2024-05-20_23-30-43",
}

In [ ]:
# Subroutines
def rescale(arr, clip=True):
    if clip:
        arr = np.clip(arr, 0 , np.percentile(arr, 99.5))
    arr -= np.min(arr)
    arr = (2**8 - 1) * (arr / np.max(arr))
    return arr.astype(np.uint8)


def get_mip(arr, axis=0, clip=True):
    mip = np.max(arr, axis=axis)
    mip = rescale(mip, clip=clip)
    return mip


def read_from_s3(img, voxel, shape, from_center=True):
    start, end = img_utils.get_start_end(voxel, shape, from_center=from_center)
    return img[0, 0, start[2]:end[2], start[1]:end[1], start[0]:end[0]]


def plot_mips(volume, prefix="", clip=True):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs_names = ["XY", "XZ", "YZ"]
    for i in range(3):
        axs[i].imshow(get_mip(volume, axis=i, clip=clip))
        axs[i].set_title(prefix + axs_names[i], fontsize=16)
        axs[i].set_xticks([])
        axs[i].set_yticks([])
    plt.tight_layout()
    plt.show()


### Opem img

In [ ]:
# Parameters
s3_bucket = "aind-open-data"
dataset = "706301"
downsample_factor = 1

# Initializations
fs = s3fs.S3FileSystem()
s3_url = f"s3://{s3_bucket}/{IMG_PREFIX[dataset]}/fused.zarr/{downsample_factor}/"

# Open img
store = s3fs.S3Map(root=s3_url, s3=fs)
img = zarr.open(store, mode='r')


### Read img

In [ ]:
# 708369
mild_sites = [
    [16169, 7755, 6470],
    [34020, 6034, 17742],
    [19188, 10270, 2936],
    [19926, 15380, 10037],
    [24279, 14285, 10730],
    [15570, 8108, 14720],
    [15221, 6903, 9094],
]
moderate_sites = [
    [20652, 15669, 9630],
    [25210, 13932, 10874],
    [32250, 11149, 17993],
]
severe_sites = [
    [18116, 17578, 17981],
]

# 706301
mild_sites = [
    [30586.32,14624.844,6969.2915],
    [17905.553,11089.657,2404.1853]
    
]
moderate_sites = [
    [20640, 8171, 11767],
    [36810.43,9475.887,19464.324],
    
]


# Doubling in x,y,z
# [36063.15,9586.425,20097.422]

In [ ]:
# Region of interest
origin = [17905.553,11089.657,2404.1853]
shape = [512, 512, 512]
from_center = True

# Read img
#shape = [shape[i] // 2 ** downsample_factor for i in range(3)]
voxel = img_utils.to_voxels(origin, downsample_factor=downsample_factor)
img_chunk = read_from_s3(img, voxel, shape, from_center=from_center)
img_chunk = rescale(img_chunk, clip=False)
imwrite("/root/capsule/data/blurred/raw2.tif", img_chunk)

# Plot mips
plot_mips(img_chunk)


In [ ]:
# Save mips
from PIL import Image


# Compute mips
mip_xz = get_mip(img_chunk, axis=1).astype(np.uint8)
mip_yz = get_mip(img_chunk, axis=2).astype(np.uint8)

# Save mips
Image.fromarray(mip_xz).save('xz.png')
Image.fromarray(mip_yz).save('yz.png')


### PSF Routines

In [ ]:
def gaussian_3d(shape, sigma, mu=[0, 0, 0]):
    x, y, z = get_coords_3d(shape)
    gaussian = (1 / (2 * np.pi * np.prod(sigma))) * \
          np.exp(- ((x - mu[0]) ** 2 / (2 * sigma[0] ** 2) +
                    (y - mu[1]) ** 2 / (2 * sigma[1] ** 2) +
                    (z - mu[2]) ** 2 / (2 * sigma[2] ** 2)))
    return gaussian / gaussian.sum()


def gaussian_peaks_3d(shape, amplitudes, sigmas, mus):
    gaussian = np.zeros(shape)
    for i in range(len(sigmas)):
        print(sigmas[i], mus[i])
        gaussian += amplitudes[i] * gaussian_3d(shape, sigmas[i], mu=mus[i])
    return gaussian


def get_coords_3d(shape):
    """
    Generates 3D coordinate grids for a given shape.

    Parameters
    ----------
    shape : Tuple/List
        Shape of the output grids.

    Returns
    -------
    tuple
        xyz coordinates.

    """
    xyz = np.indices(shape)
    for i in range(3):
        xyz[i] -= shape[i] // 2
    return xyz


def estimate_gaussian_stds(img):
    x, y = np.indices(img.shape)
    mu_x, mu_y = center_of_mass(img)
    var_x = np.sum(img * (x - mu_x)**2) / np.sum(img)
    var_y = np.sum(img * (y - mu_y)**2) / np.sum(img)
    return np.sqrt(var_x), np.sqrt(var_y)


# Initialize gaussian kernel
shape = (29, 7, 7)
sigma = [5, 1.6, 1.6]
psf_kernel = gaussian_3d(shape, sigma)

# Initialize gaussian peaks kernel
amplitudes = [2.0, 2.0, 2.0]
sigmas = [
    [1.25, 1.25, 1.25],
    [1.5, 1.25, 1.25],
    #[2, 1.25, 1.25]
]
mus = [
    [0, 0, 0],
    [-12, 0, 0],
    [-6, 0, 0]
]
psf_peaks_kernel = gaussian_peaks_3d(shape, amplitudes, sigmas, mus)
#psf_peaks_kernel[0:10, :, :] = 0

# Visualize Kernel
#plot_mips(psf_kernel, clip=False)
plot_mips(psf_peaks_kernel, clip=False)


## Generate Image Artifacts

In [ ]:
from scipy.ndimage import convolve

import torch
import torch.nn as nn


class Conv3D(nn.Module):
    
    def __init__(self, kernel):
        super(Conv3D, self).__init__()
        kernel = np.expand_dims(kernel, axis=(0, 1))
        self.conv3d = nn.Conv3d(
            in_channels=1,
            out_channels=1,
            kernel_size=kernel.shape[2:],
            stride=1,
            padding=0,
        )
        self.conv3d.weight.data = torch.tensor(kernel, dtype=torch.float32)
        self.conv3d.bias.data.zero_()

    def forward(self, x):
        return self.conv3d(x)


# pytorch convolution
t0 = time()
convolve_3d = Conv3D(psf_peaks_kernel).to("cuda")

input_tensor = torch.tensor(np.expand_dims(img_chunk, axis=(0, 1)), dtype=torch.float32).to("cuda")
convolved_img_chunk = convolve_3d(input_tensor).detach().cpu()
convolved_img_chunk = np.array(convolved_img_chunk[0, 0, ...])
print(f"Convolution Runtime: {time() - t0} seconds")

# Visualize Results
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
axes_name = ["XY", "XZ", "YZ"]
for i in range(3):
    axs[0, i].imshow(get_mip(img_chunk, axis=i))
    axs[0, i].set_title("Before - " + axes_name[i], fontsize=14)
    axs[0, i].set_xticks([])
    axs[0, i].set_yticks([])

    axs[1, i].imshow(get_mip(convolved_img_chunk, axis=i))
    axs[1, i].set_title("After - " + axes_name[i], fontsize=14)
    axs[1, i].set_xticks([])
    axs[1, i].set_yticks([])
plt.tight_layout()

## Richardson-Lucy Deconvolution

In [ ]:
# Main
t0 = time()
num_iter = 5
rl_result = restoration.richardson_lucy(convolved_img_chunk, psf_peaks_kernel, num_iter, False)
print(f"RL Runtime: {time() - t0} seconds")

# Visualize Resultss
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
axes_name = ["XY", "XZ", "YZ"]
for i in range(3):
    axs[0, i].imshow(get_mip(convolved_img_chunk, axis=i))
    axs[0, i].set_title("Before - " + axes_name[i], fontsize=14)
    axs[0, i].set_xticks([])
    axs[0, i].set_yticks([])

    axs[1, i].imshow(get_mip(rl_result, axis=i))
    axs[1, i].set_title("After - " + axes_name[i], fontsize=14)
    axs[1, i].set_xticks([])
    axs[1, i].set_yticks([])
plt.tight_layout()
plt.show()


## Wiener Deconvolution

In [ ]:
# Main
t0 = time()
wiener_result = restoration.wiener(convolved_img_chunk, psf_peaks_kernel, 3, clip=False)
wiener_result = np.clip(wiener_result, 0, np.inf)
print(f"RL Runtime: {time() - t0} seconds")

# Visualize Results
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
axes_name = ["XY", "XZ", "YZ"]
for i in range(3):
    axs[0, i].imshow(get_mip(convolved_img_chunk, axis=i))
    axs[0, i].set_title("Before - " + axes_name[i], fontsize=14)
    axs[0, i].set_xticks([])
    axs[0, i].set_yticks([])

    axs[1, i].imshow(get_mip(wiener_result, axis=i))
    axs[1, i].set_title("After - " + axes_name[i], fontsize=14)
    axs[1, i].set_xticks([])
    axs[1, i].set_yticks([])
plt.tight_layout()
plt.show()


plt.tight_layout()
plt.show()
imwrite("before.tif", img_chunk)
imwrite("wiener_result.tif", wiener_result - 15)

In [ ]:
print(np.mean(img_chunk), np.mean(wiener_result))
print(np.min(img_chunk), np.min(wiener_result))
print(np.max(img_chunk), np.max(wiener_result))